In [1]:
!pip install --upgrade tensorflow tensorflow-hub tensorflow-datasets pandas scikit-learn

  Using cached tensorflow-2.21.0-cp313-cp313-manylinux_2_27_x86_64.whl.metadata (4.4 kB)


In [2]:
!pip install numpy
!pip install tensorflow
!pip install tensorflow-hub
!pip install tensorflow-datasets
!pip install pandas
!pip install scikit-learn

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_datasets as tfds

/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [4]:
df = pd.read_csv('Spam_SMS.csv')

In [111]:
df['Class'] = df['Class'].apply(lambda x: 1 if x == 'spam' else 0)

In [112]:
features = df['Message'].values
labels = df['Class'].values

In [113]:
# Converted the features and labels into standard NumPy arrays to prevent indexing issues arising from PyArrow-backed data types.
features_for_split = np.asarray(features)
labels_for_split = np.asarray(labels)

train_features, temp_features, train_labels, temp_labels = train_test_split(features_for_split, labels_for_split, test_size=0.4, random_state=42, stratify=labels_for_split)

In [114]:
validation_features, test_features, validation_labels, test_labels = train_test_split(temp_features, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels)

In [115]:
train_data = tf.data.Dataset.from_tensor_slices((train_features, train_labels))
validation_data = tf.data.Dataset.from_tensor_slices((validation_features, validation_labels))
test_data = tf.data.Dataset.from_tensor_slices((test_features, test_labels))

In [116]:
print(f"Training samples: {len(train_features)}")
print(f"Validation samples: {len(validation_features)}")
print(f"Test samples: {len(test_features)}")

Training samples: 3344
Validation samples: 1115
Test samples: 1115


In [117]:
train_data

<_TensorSliceDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>

In [118]:
print("GPU is", "available" if tf.config.experimental.list_physical_devices("GPU") else "NOT AVAILABLE")

GPU is NOT AVAILABLE


In [119]:
train_example_batch, train_labels_batch = next(iter(train_data.batch(10)))

In [120]:
train_example_batch

<tf.Tensor: shape=(10,), dtype=string, numpy=
array([b"Didn't try, g and I decided not to head out",
       b'K.k.this month kotees birthday know?',
       b'FREE for 1st week! No1 Nokia tone 4 ur mobile every week just txt NOKIA to 8077 Get txting and tell ur mates. www.getzed.co.uk POBox 36504 W45WQ 16+ norm150p/tone',
       b'TheMob> Check out our newest selection of content, Games, Tones, Gossip, babes and sport, Keep your mobile fit and funky text WAP to 82468',
       b'Not getting anywhere with this damn job hunting over here!',
       b'Ok but tell me half an hr b4 u come i need 2 prepare.',
       b'Last chance 2 claim ur \xc2\xa3150 worth of discount vouchers-Text YES to 85023 now!SavaMob-member offers mobile T Cs 08717898035. \xc2\xa33.00 Sub. 16 . Remove txt X or STOP',
       b'SIX chances to win CASH! From 100 to 20,000 pounds txt> CSH11 and send to 87575. Cost 150p/day, 6days, 16+ TsandCs apply Reply HL 4 info',
       b"Nooooooo I'm gonna be bored to death all day. Cab

In [121]:
train_labels_batch

<tf.Tensor: shape=(10,), dtype=int64, numpy=array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])>

In [122]:
embedding= "https://tfhub.dev/google/nnlm-en-dim50/2"
hub_layer = hub.KerasLayer(embedding, input_shape=[], dtype=tf.string, trainable=True)

In [123]:
hub_layer(train_example_batch[:3])

<tf.Tensor: shape=(3, 50), dtype=float32, numpy=
array([[ 0.11804529, -0.20439024, -0.15273513,  0.3631096 ,  0.2639256 ,
         0.1427441 ,  0.0154772 , -0.11546327, -0.06032599,  0.12595573,
         0.01730763,  0.16129725,  0.13050973, -0.02411952, -0.09265824,
         0.11918097,  0.01606453,  0.00073031,  0.08658984, -0.35841978,
         0.09853654,  0.11273977,  0.29499108,  0.12503691,  0.03247668,
         0.3039198 , -0.27432537, -0.0140052 , -0.09324268, -0.14914821,
         0.08100492,  0.04808869,  0.19345912, -0.09598191, -0.17675653,
         0.09128913, -0.15570243, -0.11658414,  0.08024584, -0.30807707,
         0.05655973,  0.28446248,  0.08266633, -0.02424893, -0.12911971,
         0.18448065, -0.14906633, -0.22766146, -0.20790046,  0.19208291],
       [ 0.42215577, -0.03116323, -0.06193922, -0.04012592,  0.28326538,
        -0.06097605,  0.11285309,  0.12880908,  0.14121933,  0.08997889,
         0.18266648, -0.23275085,  0.02760297,  0.00408503,  0.22547741,
 

#**Creating the Sequential Model**

Encountered a ValueError while building the model in Google Colab, caused by a version incompatibility between the installed TensorFlow and TensorFlow Hub packages.

The root cause was that tensorflow_hub.keras_layer.KerasLayer wasn't being added to the Sequential model correctly. I resolved this by wrapping the hub.KerasLayer instance inside a tf.keras.layers.Lambda layer, making the Hub layer compatible with the Sequential architecture in this Keras version. After applying the fix, model summary executed successfully, confirming the architecture was built correctly.

In [124]:
model = tf.keras.Sequential([
    tf.keras.layers.Lambda(lambda x: hub_layer(x), dtype=tf.string),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1)
])
model.build((None,))

In [125]:
model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda_7 (Lambda)               │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 16)             │           816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 833 (3.25 KB)

 Trainable params: 833 (3.25 KB)

 Non-trainable params: 0 (0.00 B)

In [126]:
model.compile(optimizer='adam', loss=tf.keras.losses.BinaryCrossentropy(from_logits=True), metrics=['accuracy'])

In [127]:
history = model.fit(train_data.shuffle(10000).batch(100), epochs=75, validation_data=validation_data.batch(100), verbose=1)

Epoch 1/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9946 - loss: 0.6937 - val_accuracy: 1.0000 - val_loss: 0.5813
Epoch 2/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.4993 - val_accuracy: 1.0000 - val_loss: 0.4014
Epoch 3/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.3363 - val_accuracy: 1.0000 - val_loss: 0.2588
Epoch 4/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.2173 - val_accuracy: 1.0000 - val_loss: 0.1667
Epoch 5/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.1446 - val_accuracy: 1.0000 - val_loss: 0.1141
Epoch 6/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.1022 - val_accuracy: 1.0000 - val_loss: 0.0826
Epoch 7/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0761 - val_accuracy: 1.0000 - val_loss: 0.0627
Epoch 8/75
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0591 - val_accuracy: 1.0000 - val_loss

In [128]:
results = model.evaluate(test_data.batch(100), verbose=2)

for name, value in zip(model.metrics_names, results):
  print("%s: %.3f" % (name, value))

12/12 - 0s - 7ms/step - accuracy: 1.0000 - loss: 0.0013
loss: 0.001
compile_metrics: 1.000
